In [1]:
# automatically reload imported modules before executing code

%load_ext autoreload
%autoreload 2

# Model Inference

Load saved models and make predictions on songs with metadata.

In [16]:
from pyrekordbox import Rekordbox6Database
import polars as pl
from nbutils import setup_path

setup_path()
db = Rekordbox6Database()

pl.Config.set_tbl_rows(30)  # Show 100 row

[10:06:38] pyrekordbox.db6.database:WARNING  - Rekordbox is running!


polars.config.Config

In [17]:
# Load saved models
from models import load_model
from pathlib import Path

models_dir = Path("../models")

# Find latest models
rf_model_file = sorted(models_dir.glob("random_forest_Genre_*_model.pkl"))[-1]
xgb_model_file = sorted(models_dir.glob("xgboost_Genre_*_model.pkl"))[-1]

print(f"Loading: {rf_model_file.name}")
rf_data = load_model(str(rf_model_file))

print(f"\nLoading: {xgb_model_file.name}")
xgb_data = load_model(str(xgb_model_file))

Loading: random_forest_Genre_20251015_232027_model.pkl

MODEL LOADED SUCCESSFULLY
Model name: random_forest_Genre
Tag group: Genre
Number of labels: 22
Saved on: 2025-10-15T23:20:27.523168

Loaded components:
  Model: ✓
  Thresholds: ✓
  Scaler: ✓
  PCA: ✗

Stored metrics:
  macro_f1: 0.4909
  macro_precision: 0.4601
  macro_recall: 0.6430


Loading: xgboost_Genre_20251015_232027_model.pkl

MODEL LOADED SUCCESSFULLY
Model name: xgboost_Genre
Tag group: Genre
Number of labels: 22
Saved on: 2025-10-15T23:20:27.538256

Loaded components:
  Model: ✓
  Thresholds: ✓
  Scaler: ✓
  PCA: ✗

Stored metrics:
  macro_f1: 0.5060
  macro_precision: 0.4732
  macro_recall: 0.6371



In [18]:
# Get songs and features  
from utils import get_clean_songs
from processing import ProcessingResult

songs_df = get_clean_songs(db, rename=True)
features_df = pl.read_parquet("../data/song_features.parquet")

# Define feature columns (same as training)
exclude_cols = [
    "song_path", "harmonic_percussive_ratio", "percussive_strength",
    "tonnetz_mean_0", "tonnetz_mean_1", "tonnetz_mean_2", "tonnetz_mean_3", "tonnetz_mean_4", "tonnetz_mean_5",
    "tonnetz_std_0", "tonnetz_std_1", "tonnetz_std_2", "tonnetz_std_3", "tonnetz_std_4", "tonnetz_std_5"
]
feature_cols = [col for col in features_df.columns if col not in exclude_cols + ["song_id", "song_path"]]

# Filter features
features_df = features_df.filter(pl.col("energy_increase_ratio").is_not_null())
X_test = features_df.select(["song_id"] + feature_cols)

# Scale features
X_test_scaled = rf_data['scaler'].transform(X_test.drop("song_id"))
X_test_scaled = pl.concat([X_test.select("song_id"), X_test_scaled], how="horizontal")

# Create ProcessingResult for predict_with_metadata
processing_result = ProcessingResult(
    X_train=None, X_val=None, X_test=X_test_scaled,
    y_train=None, y_val=None, y_test=None,
    tags=rf_data['tags'],
    song_ids_train=None, song_ids_val=None, song_ids_test=X_test.select("song_id"),
    scaler=rf_data['scaler'], pca=None
)

print(f"Prepared {X_test_scaled.shape[0]} songs for inference")

Prepared 515 songs for inference


In [9]:
# Make predictions with BOTH models at once (with optimized thresholds!)
from processing import predict_with_optimized_thresholds
from nbutils import display_polars

predictions = predict_with_optimized_thresholds(
    models={
        "Random Forest": rf_data['model'],
        "XGBoost": xgb_data['model']
    },
    X_test=processing_result.X_test,
    tags=rf_data['tags'],  # Same tags for both
    thresholds={
        "Random Forest": rf_data['thresholds'],
        "XGBoost": xgb_data['thresholds']
    },
    songs_df=songs_df
)

In [11]:
predictions.schema 

Schema([('song_id', String),
        ('song_path', String),
        ('song_title', String),
        ('artist_id', String),
        ('artist_name', String),
        ('genre_id', String),
        ('genre_name', String),
        ('bpm', Int32),
        ('date_created', String),
        ('length', Int32),
        ('tag_names', List(String)),
        ('tag_ids', List(String)),
        ('sample_rate', Int32),
        ('has_tags', Boolean),
        ('Random Forest_predicted_tags', List(String)),
        ('XGBoost_predicted_tags', List(String))])

In [15]:
import polars as pl

display_df = (
    predictions
    .select(
        pl.col("song_title"),
        pl.col("artist_name"),
        pl.col( "tag_names"),
        pl.col("Random Forest_predicted_tags").alias("rf_preds"),
        pl.col("XGBoost_predicted_tags").alias("xgb_preds")
    )
    .with_columns(
        # Intersection: elements in both lists
        pl.col("rf_preds").list.set_intersection("xgb_preds").list.len().alias("intersection"),
        # Union: unique elements from both lists
        pl.col("rf_preds").list.set_union("xgb_preds").list.len().alias("union")
    )
    .with_columns(
        # Jaccard = intersection / union
        (pl.col("intersection") / pl.col("union")).fill_null(1.0).alias("agreement")
    )
)

display_polars(display_df, 50, 1)

shape: (50, 8)
┌─────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬───────┬───────────┐
│ song_title  ┆ artist_nam ┆ tag_names  ┆ rf_preds   ┆ xgb_preds  ┆ intersecti ┆ union ┆ agreement │
│ ---         ┆ e          ┆ ---        ┆ ---        ┆ ---        ┆ on         ┆ ---   ┆ ---       │
│ str         ┆ ---        ┆ list[str]  ┆ list[str]  ┆ list[str]  ┆ ---        ┆ u32   ┆ f64       │
│             ┆ str        ┆            ┆            ┆            ┆ u32        ┆       ┆           │
╞═════════════╪════════════╪════════════╪════════════╪════════════╪════════════╪═══════╪═══════════╡
│ Save Our    ┆ Escape     ┆ ["80s",    ┆ ["Disco",  ┆ ["Disco",  ┆ 5          ┆ 5     ┆ 1.0       │
│ Love        ┆ From New   ┆ "Disco",   ┆ "Funk",    ┆ "Funk",    ┆            ┆       ┆           │
│             ┆ York       ┆ "Funk",    ┆ "Motown",  ┆ "Motown",  ┆            ┆       ┆           │
│             ┆            ┆ "Soul",    ┆ "Pop",     ┆ "Pop",     ┆         